# 4 Feature Selection and Engineering

Notes for start

- Remove irrelevant or highly correlated features
- Use domain logic: e.g., create new features like **Spending Score**, **Frequency Index**
- Scale features using StandardScaler or similar

---
**Phase 1: The Core Foundation (RFM Engineering)**

Almost all e-commerce customer segmentation starts with the industry-standard RFM model. You will need to aggregate your transaction data by CustomerID.
- Recency: How many days has it been since the customer's last purchase? (Calculated against a "snapshot date," usually the day after the last transaction in your dataset).
- Frequency: How many distinct orders (invoices) has this customer placed?
- Monetary: What is the total lifetime revenue generated by this customer? (Sum of quantity $\times$ unit_price).

**Phase 2: Advanced Behavioral Features (Leveraging Your EDA)**

This is where you make your segmentation unique by using the categories you just finished building in your EDA. You can create percentage-based features for each customer to capture how they shop.
- Pricing Preferences:
    - % Spend on Low-Ticket (What percentage of their total spend is on items $\le \$5.00$?);
    - % Spend on Premium (What percentage is on items $> \$10.00$?)
- Customer Volume Persona (B2B vs. B2C):
    - Create a binary flag (is_b2b) if a significant portion of their orders falls into your "Qty 13–120" or higher tiers.
    - Average order size (Total items bought / Total invoices).
- Geographic Flags:
    - is_domestic (1 if UK, 0 if International). Since the UK is ~90% of your data, splitting by specific country might create too much noise, but a Domestic vs. Export flag is highly valuable.
- Engagement Metrics:
    - Average Order Value (AOV): Total spend / Total invoices.
    - Return Rate: If your dataset includes negative quantities (cancellations/returns), calculate the ratio of returned items to purchased items.

**Phase 3: Feature Transformation (Pre-processing for ML)** - Leave for stage 5 - Unsupervised Learning – Clustering

Clustering algorithms (like K-Means) are heavily distance-based. If you feed them raw financial data, they will perform poorly. You must prepare the data mathematically.
- Handle Skewness: Financial data (Monetary, Frequency) is almost always heavily right-skewed (a few whales spending millions, thousands of users spending $10). You will likely need to apply a Log Transformation to normalize these distributions.
- Standardization (Scaling): Because "Recency" is measured in days (e.g., 15 to 300) and "Monetary" is measured in dollars (e.g., $50 to $15,000), you must scale the features so they have equal weight. Use StandardScaler to give every feature a mean of 0 and a standard deviation of 1.

**Phase 4: Feature Selection & Dimensionality Reduction**

You don't want to feed the model 30 overlapping features. Too much noise will ruin the clusters.
- Correlation Analysis: Build a correlation matrix (heatmap) of your newly engineered customer features. If two features are highly correlated (e.g., "Total Items Bought" and "Monetary Value" will likely have a 0.90+ correlation), drop one to avoid multicollinearity.
- Dimensionality Reduction (Optional but recommended): If you end up with 10+ features (e.g., various product category preferences, RFM, geography), consider running Principal Component Analysis (PCA) to compress these features into 2 or 3 principal components before clustering.

**Phase 5: Modeling Preparation**

Once your final, scaled customer matrix is ready, you will move to the actual modeling phase.
- Algorithm Selection: Determine if you will use K-Means (best for general customer tiers) or DBSCAN (great if you have a lot of weird outliers you want to isolate).
- Finding "K": Plan to use the Elbow Method and Silhouette Scores on your engineered dataset to find the optimal number of customer segments.
---

What is Feature Engineering?
Feature engineering is the process of creating new features or transforming existing ones to make them more useful for your model.



---

Simple to-do plan
- 4.1 Reshaping to customer grain
- 4.2 Core RFM construction
- 4.3 Recovering the cancellations dataset
- 4.4 Extended behavioral features
- 4.5 Skew & outlier treatment
- 4.6 Categorical encoding
- 4.7 Feature scaling
- 4.8 Feature selection / dimensionality check
- 4.9 Final feature table assembly & validation
- 4.10 Results of Feature Selection & Engineering (mirroring your 3.4 conclusions section)

or

1. Feature Audit
2. Remove useless variables
3. Feature Transformation
4. Feature Engineering
5. Feature Selection
6. Scaling
7. Dimensionality Reduction (optional)
8. Final Modeling Dataset



In [1]:
import numpy as np 
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent
raw_data_directory_path = project_root / "data" / "raw"
processed_data_directory_path = project_root / "data" / "processed"

In [3]:
df = pd.read_csv(processed_data_directory_path / "e-commerce-data-cleaned.csv", encoding = "ISO-8859-1")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 391068 entries, 0 to 391067
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   invoice_number    391068 non-null  int64  
 1   stock_code        391068 non-null  str    
 2   description       391068 non-null  str    
 3   quantity          391068 non-null  int64  
 4   invoice_datetime  391068 non-null  str    
 5   invoice_date      391068 non-null  str    
 6   invoice_time      391068 non-null  str    
 7   unit_price        391068 non-null  float64
 8   customer_id       391068 non-null  float64
 9   country           391068 non-null  str    
dtypes: float64(2), int64(2), str(6)
memory usage: 60.4 MB


In [4]:
df_canceled = pd.read_csv(processed_data_directory_path / "e-commerce-data-canceled.csv", encoding = "ISO-8859-1")
df_canceled.info()

<class 'pandas.DataFrame'>
RangeIndex: 8872 entries, 0 to 8871
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   invoice_number    8872 non-null   str    
 1   stock_code        8872 non-null   str    
 2   description       8872 non-null   str    
 3   quantity          8872 non-null   int64  
 4   invoice_datetime  8872 non-null   str    
 5   invoice_date      8872 non-null   str    
 6   invoice_time      8872 non-null   str    
 7   unit_price        8872 non-null   float64
 8   customer_id       8872 non-null   float64
 9   country           8872 non-null   str    
dtypes: float64(2), int64(1), str(7)
memory usage: 1.4 MB
